In [3]:
import os
os.environ["CALITP_BQ_MAX_BYTES"] = str(800_000_000_000) ## 800GB?

from calitp.tables import tbl
from calitp import query_sql
import calitp.magics

import shared_utils
# import utils

from siuba import *
import pandas as pd
import geopandas as gpd
# import shapely

import datetime as dt
import time
# from zoneinfo import ZoneInfo

# import gcsfs
# fs = gcsfs.GCSFileSystem()

from tqdm import tqdm_notebook
from tqdm.notebook import trange, tqdm

In [4]:
df = query_sql(f"""
SELECT *
FROM `cal-itp-data-infra-staging.eric_staging.reports_daily_service_routes_stops`
""")

In [5]:
df

,feed_key,calitp_itp_id,calitp_url_number,service_date,ttl_service_hours,n_trips,n_routes,first_departure_ts,last_arrival_ts,n_stops
0,-3807002727485986157,350,0,2022-05-22,45.300000,102,5,25860,69600,159
1,-637281990126960778,350,0,2022-05-29,45.300000,102,5,25860,69600,159
2,-3807002727485986157,350,0,2022-05-15,45.300000,102,5,25860,69600,159
3,5416755288360399371,350,0,2022-05-08,45.300000,102,5,25860,69600,159
4,5416755288360399371,350,0,2022-05-01,45.300000,102,5,25860,69600,159
...,...,...,...,...,...,...,...,...,...,...
85,-947628124198347129,350,2,2022-05-21,50.316667,116,5,25440,70500,210
86,-947628124198347129,350,2,2022-05-14,50.316667,116,5,25440,70500,210
87,-947628124198347129,350,2,2022-05-07,50.316667,116,5,25440,70500,210
88,8947966922620810233,350,2,2022-05-28,50.316667,116,5,25440,70500,210


In [3]:
CALITP_ITP_ID = 350 ## Union City
CALITP_URL_NUMBER = 0 ## think this is right...

In [4]:
## current reports code

ids_with_feeds = (
    tbl.views.reports_gtfs_schedule_index()
    # >> filter(_.use_for_report)
    >> filter(_.date_start == '2022-05-01')
    >> filter(_.calitp_itp_id == CALITP_ITP_ID)
    >> collect()
)

In [5]:
ids_with_feeds >> arrange(-_.date_start)

,publish_date,calitp_itp_id,calitp_url_number,agency_name,date_start,date_end,has_feed_info,is_private_feed,use_for_report
0,2022-06-01,350,2,Union City Transit,2022-05-01,2022-05-31,True,False,False
1,2022-06-01,350,1,Union City Transit,2022-05-01,2022-05-31,True,False,False
2,2022-06-01,350,0,Union City Transit,2022-05-01,2022-05-31,True,False,True


In [6]:
filter_itp = filter(
    _.calitp_itp_id == CALITP_ITP_ID, _.calitp_url_number == CALITP_URL_NUMBER
)

In [7]:
DATE_START = dt.date(2022, 5, 1)
DATE_END = dt.date(2022, 5, 31)

In [8]:
BIWEEKLY_MARKERS = pd.date_range(DATE_START, DATE_END, freq="2W").astype(str).tolist()
# BIWEEKLY_MARKERS = pd.date_range(DATE_START, DATE_END, freq="2W").tolist()

## otherwise errors, fix this on main reports? ask Natalya?
BIWEEKLY_MARKERS = [dt.date.fromisoformat(marker) for marker in BIWEEKLY_MARKERS]

In [9]:
## current reports code

# Analyze validation notices ----

from sqlalchemy.sql import func

tbl_biweekly = (
    tbl.views.dim_date()
    >> filter(_.full_date.isin(BIWEEKLY_MARKERS))
    >> select(_.date == _.full_date)
)

In [10]:
tbl_validation_notices_raw = (
    tbl.gtfs_schedule_type2.validation_notices()
    >> filter_itp
    >> inner_join(
        _,
        tbl_biweekly,
        sql_on = (lambda lhs, rhs:
                  (lhs.calitp_extracted_at <= rhs.date) &
                  (func.coalesce(lhs.calitp_deleted_at, "2099-01-01") > rhs.date)
        )
    )
    # do a simple count of the number of notices on each date
    >> select(_.date, _.code, _.severity)
    >> count(_.date, _.code, _.severity)
    >> collect()
)

In [11]:
tbl_validation_notices_raw

,date,code,severity,n
0,2022-05-01,leading_or_trailing_whitespaces,ERROR,40
1,2022-05-29,leading_or_trailing_whitespaces,ERROR,40
2,2022-05-15,leading_or_trailing_whitespaces,ERROR,40
3,2022-05-01,decreasing_or_equal_shape_distance,ERROR,6
4,2022-05-29,decreasing_or_equal_shape_distance,ERROR,6
5,2022-05-15,decreasing_or_equal_shape_distance,ERROR,6
6,2022-05-29,route_short_and_long_name_equal,WARNING,5
7,2022-05-15,route_short_and_long_name_equal,WARNING,5
8,2022-05-01,route_short_and_long_name_equal,WARNING,5
9,2022-05-15,same_name_and_description_for_route,ERROR,4


In [39]:
# create a DataFrame with correct columns and no rows
_validation_empty = pd.DataFrame(
    dict([("code", []), ("severity", []), *zip(BIWEEKLY_MARKERS, [tuple()] * len(BIWEEKLY_MARKERS))]),
)

# spread validation notices so dates are columns, and counts are values
# TODO: when we spread the dates out, the counts sometimes can become floats,
# so have decimals, which might look funky. We can fix this by either converting
# the counts to strings before the code below
if not tbl_validation_notices_raw.shape[0]:
    _tbl_validation_notices = _validation_empty
else:
    out_wide = (
        tbl_validation_notices_raw >> mutate(date=_.date.astype(str)) >> spread(_.date, _.n)
    )
    
    _tbl_validation_notices = pd.concat(
        [_validation_empty, out_wide], ignore_index=True
    ).fillna("")

In [40]:
out_wide

,code,severity,2022-05-01,2022-05-15,2022-05-29
0,decreasing_or_equal_shape_distance,ERROR,6.0,6.0,6.0
1,leading_or_trailing_whitespaces,ERROR,40.0,40.0,40.0
2,missing_feed_info_date,WARNING,NaN,1.0,1.0
3,route_short_and_long_name_equal,WARNING,5.0,5.0,5.0
4,same_name_and_description_for_route,ERROR,4.0,4.0,4.0
5,unknown_file,INFO,3.0,3.0,3.0


In [14]:
## deprecated table?? ask...
# tbl.views.validation_notices()

In [15]:
tbl.views.validation_fact_daily_feed_notices() >> head(3)

,feed_key,date,validation_created_at,validation_deleted_at,calitp_itp_id,calitp_url_number,code,severity,csvRowNumber,oldCsvRowNumber,...,routeLongName,routeDesc,stopId,stopName,serviceIdA,serviceIdB,departureTime,arrivalTime,parentStation,parentStopName
0,-3540410672069212686,2022-04-28,2022-04-28,2022-04-29,105,0,same_name_and_description_for_route,ERROR,11,None,...,None,Route 12,None,None,None,None,None,None,None,None
1,1273942275370395959,2022-06-02,2022-06-02,2022-06-03,231,0,same_name_and_description_for_route,ERROR,32,None,...,None,60-Hwy126,None,None,None,None,None,None,None,None
2,4493221564916387693,2022-05-03,2022-05-03,2022-05-04,361,0,same_name_and_description_for_route,ERROR,4,None,...,None,Route 2,None,None,None,None,None,None,None,None


In [29]:
# tbl_validation_notices_better = (
#     tbl.views.validation_fact_daily_feed_notices()
#     >> filter_itp
#     >> inner_join(
#         _,
#         tbl_biweekly,
#         sql_on = (lambda lhs, rhs:
#                   (lhs.validation_created_at <= rhs.date) &
#                   (func.coalesce(lhs.validation_deleted_at, "2099-01-01") > rhs.date)
#         )
#     )
# )

In [30]:
# val_notices = tbl_validation_notices_better >> collect()

In [44]:
BIWEEKLY_MARKERS

[datetime.date(2022, 5, 1),
 datetime.date(2022, 5, 15),
 datetime.date(2022, 5, 29)]

In [31]:
## this is so much easier+better

test2 = (tbl.views.validation_fact_daily_feed_notices()
         >> filter_itp
         >> filter(_.date.isin(BIWEEKLY_MARKERS))
        ) >> collect()

In [37]:
# create a DataFrame with correct columns and no rows
_validation_empty = pd.DataFrame(
    dict([("code", []), ("severity", []), *zip(BIWEEKLY_MARKERS, [tuple()] * len(BIWEEKLY_MARKERS))]),
)

# spread validation notices so dates are columns, and counts are values
# TODO: when we spread the dates out, the counts sometimes can become floats,
# so have decimals, which might look funky. We can fix this by either converting
# the counts to strings before the code below
if not test2.shape[0]:
    _tbl_validation_notices = _validation_empty
else:
    out_wide_2 = (test2
        >> select(_.date, _.code, _.severity)
        >> count(_.date, _.code, _.severity)
        >> mutate(date=_.date.astype(str)) >> spread(_.date, _.n)
    )
    
    _tbl_validation_notices = pd.concat(
        [_validation_empty, out_wide], ignore_index=True
    ).fillna("")

In [41]:
out_wide

,code,severity,2022-05-01,2022-05-15,2022-05-29
0,decreasing_or_equal_shape_distance,ERROR,6.0,6.0,6.0
1,leading_or_trailing_whitespaces,ERROR,40.0,40.0,40.0
2,missing_feed_info_date,WARNING,NaN,1.0,1.0
3,route_short_and_long_name_equal,WARNING,5.0,5.0,5.0
4,same_name_and_description_for_route,ERROR,4.0,4.0,4.0
5,unknown_file,INFO,3.0,3.0,3.0


In [38]:
out_wide_2

,code,severity,2022-05-01,2022-05-15,2022-05-29
0,decreasing_or_equal_shape_distance,ERROR,6.0,6.0,6.0
1,leading_or_trailing_whitespaces,ERROR,40.0,40.0,40.0
2,missing_feed_info_date,WARNING,NaN,1.0,1.0
3,route_short_and_long_name_equal,WARNING,5.0,5.0,5.0
4,same_name_and_description_for_route,ERROR,4.0,4.0,4.0
5,unknown_file,INFO,3.0,3.0,3.0


In [20]:
test2.code.unique()

array(['unknown_file', 'same_name_and_description_for_route',
       'leading_or_trailing_whitespaces',
       'decreasing_or_equal_shape_distance',
       'route_short_and_long_name_equal', 'missing_feed_info_date'],
      dtype=object)

In [21]:
test2 >> count(_.date, _.code)

,date,code,n
0,2022-05-01,decreasing_or_equal_shape_distance,6
1,2022-05-01,leading_or_trailing_whitespaces,40
2,2022-05-01,route_short_and_long_name_equal,5
3,2022-05-01,same_name_and_description_for_route,4
4,2022-05-01,unknown_file,3
5,2022-05-15,decreasing_or_equal_shape_distance,6
6,2022-05-15,leading_or_trailing_whitespaces,40
7,2022-05-15,missing_feed_info_date,1
8,2022-05-15,route_short_and_long_name_equal,5
9,2022-05-15,same_name_and_description_for_route,4


In [22]:
disp = test2 >> filter(_.code != 'leading_or_trailing_whitespaces')

In [23]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(disp >> head(15))

,feed_key,date,validation_created_at,validation_deleted_at,calitp_itp_id,calitp_url_number,code,severity,csvRowNumber,oldCsvRowNumber,newCsvRowNumber,csvRowNumberA,csvRowNumberB,rowLength,headerCount,previousCsvRowNumber,filename,fieldName,fieldName1,fieldName2,fieldValue,fieldValue1,fieldValue2,index,shapeDistTraveled,shapePtSequence,prevCsvRowNumber,prevShapeDistTraveled,prevShapePtSequence,duplicatedField,suggestedExpirationDate,suggestedExpirationDate.localDate,suggestedExpirationDate.localDate.year,suggestedExpirationDate.localDate.month,suggestedExpirationDate.localDate.day,stopSequence,specifiedField,prevStopTimeDistTraveled,prevStopSequence,speedkmh,firstStopSequence,lastStopSequence,stopShapeThresholdMeters,blockId,intersection,expectedLocationType,locationType,parentCsvRowNumber,parentLocationType,entityCount,fieldType,childFieldName,parentFieldName,childFilename,parentFilename,fareId,previousFareId,shapeId,routeId,currentDate,feedEndDate,routeColor,routeTextColor,tripId,tripIdA,tripIdB,routeShortName,routeLongName,routeDesc,stopId,stopName,serviceIdA,serviceIdB,departureTime,arrivalTime,parentStation,parentStopName
0,-637281990126960778,2022-05-29,2022-05-29,2022-05-30,350,0,unknown_file,INFO,NaN,None,None,None,None,None,None,None,realtime_routes.txt,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,-3807002727485986157,2022-05-15,2022-05-15,2022-05-16,350,0,same_name_and_description_for_route,ERROR,3.0,None,None,None,None,None,None,None,routes.txt,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,route_short_name,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,3495,None,None,None,None,None,None,None,None,None,2Whipple,None,None,None,None,None,None,None,None
6,-3807002727485986157,2022-05-15,2022-05-15,2022-05-16,350,0,decreasing_or_equal_shape_distance,ERROR,570.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,7802.955176,187.0,569.0,7802.955176,186.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,17848,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
7,-3807002727485986157,2022-05-15,2022-05-15,2022-05-16,350,0,route_short_and_long_name_equal,WARNING,6.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,3650,None,None,None,None,None,None,None,4Tamarack,4Tamarack,None,None,None,None,None,None,None,None,None
8,-3807002727485986157,2022-05-15,2022-05-15,2022-05-16,350,0,decreasing_or_equal_shape_distance,ERROR,786.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,44.409337,3.0,785.0,44.409337,2.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,17847,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
11,5416755288360399371,2022-05-01,2022-05-01,2022-05-02,350,0,decreasing_or_equal_shape_distance,ERROR,881.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,4160.496841,98.0,880.0,4160.496841,97.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,17847,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
12,-3807002727485986157,2022-05-15,2022-05-15,2022-05-16,350,0,same_name_and_description_for_route,ERROR,6.0,None,No

In [50]:
## why doesn't the first row have the shortname and longnames??
## would rather fix in warehouse than reconstruct here...
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(disp >> filter(_.routeId == '3650') >> select(_.code, _.routeId, _.routeShortName,
                                                          _.routeLongName, _.routeDesc, _.csvRowNumber))

,code,routeId,routeShortName,routeLongName,routeDesc,csvRowNumber
7,route_short_and_long_name_equal,3650,4Tamarack,4Tamarack,None,6.0
12,same_name_and_description_for_route,3650,None,None,4Tamarack,6.0
30,same_name_and_description_for_route,3650,None,None,4Tamarack,6.0
39,same_name_and_description_for_route,3650,None,None,4Tamarack,6.0
41,route_short_and_long_name_equal,3650,4Tamarack,4Tamarack,None,6.0
140,route_short_and_long_name_equal,3650,4Tamarack,4Tamarack,None,6.0


### Some Ideas
* need for some sort of mapping code -> relevant columns...
    * some have routeId, some have fieldName, fieldValue etc...
* always extract relevant info cols -- csv row at the very least?
* shape construction and display should be fairly straightforward...

### Design
* will we need some sort of drop down/show more info tab?